# 03 · Silver — limpeza e padronização

Aplica à `bronze.vb_matches` os tratamentos definidos no perfilamento (notebook 02): `"NA"` → NULL, tipagem, altura em cm, ranking separado em seed principal/qualificatória, placar parseado, idade recalculada, flags de exceção **sem excluir linhas**. Sai em três tabelas com grãos distintos: `silver.partida`, `silver.desempenho_atleta`, `silver.atleta`.

## 1. "NA" → NULL

In [0]:
from pyspark.sql import functions as F

bronze = spark.table("workspace.bronze.vb_matches")
colunas = [c for c in bronze.columns if not c.startswith("_")]

# toda ausência na fonte é o texto "NA"; aqui vira NULL de verdade
df = bronze.select(*[F.when(F.col(c) == "NA", None).otherwise(F.col(c)).alias(c) for c in colunas])

print("nulos reais em 'round' agora:", df.filter(F.col("round").isNull()).count(), "(esperado 4.939)")

## 2. Chave da partida e colunas derivadas

In [0]:
chave_partida = ["circuit", "tournament", "year", "date", "gender",
                 "w_player1", "w_player2", "l_player1", "l_player2",
                 "bracket", "round", "match_num"]

# ranking: "7" → principal 7 | "Q16" → qualificatória 16 | "10, Q1" → 10 e 1
def seed_principal(col):
    return F.expr(f"try_cast(get(filter(split({col}, ', '), x -> x not rlike '^Q'), 0) as int)")

def seed_qualificatoria(col):
    return F.expr(f"try_cast(regexp_replace(get(filter(split({col}, ', '), x -> x rlike '^Q'), 0), 'Q', '') as int)")

placar_regular = F.col("score").rlike(r"^(\d+-\d+)(, \d+-\d+)*$")
sets_vencedor = F.expr("size(filter(split(score, ', '), s -> try_cast(split(s, '-')[0] as int) > try_cast(split(s, '-')[1] as int)))")
sets_perdedor = F.expr("size(filter(split(score, ', '), s -> try_cast(split(s, '-')[0] as int) < try_cast(split(s, '-')[1] as int)))")

duracao_min = F.expr("try_cast(split(duration, ':')[0] as int) * 60 + try_cast(split(duration, ':')[1] as int)")

fase = (F.when(F.col("bracket").rlike("^Pool"), "grupos")
         .when(F.col("bracket").rlike("^Qualifier|^Country Quota"), "qualificatoria")
         .otherwise("eliminatoria"))

base = (
    df.withColumn("id_partida", F.xxhash64(*chave_partida))
      .withColumn("data", F.to_date("date"))
      .withColumn("ano", F.col("year").cast("int"))
      .withColumn("num_partida", F.col("match_num").cast("int"))
      .withColumn("fase", fase)
      .withColumn("placar_regular", placar_regular)
      .withColumn("sets_vencedor", F.when(placar_regular, sets_vencedor))
      .withColumn("sets_perdedor", F.when(placar_regular, sets_perdedor))
      .withColumn("duracao_min", duracao_min)
      .withColumn("flag_partida_incompleta", ~placar_regular | F.col("score").isNull())
      .withColumn("flag_duracao_suspeita", F.col("duracao_min") < 15)
      # placar regular mas contraditório: 3 sets do vencedor, ou perdedor com mais sets
      .withColumn("flag_placar_inconsistente",
                  placar_regular & ~((F.col("sets_vencedor") > F.col("sets_perdedor"))
                                     & (F.col("sets_vencedor") <= 2)))
      .withColumn("w_seed_principal", seed_principal("w_rank"))
      .withColumn("w_seed_qualificatoria", seed_qualificatoria("w_rank"))
      .withColumn("l_seed_principal", seed_principal("l_rank"))
      .withColumn("l_seed_qualificatoria", seed_qualificatoria("l_rank"))
)

# conferência rápida das derivações
display(base.select("score", "placar_regular", "sets_vencedor", "sets_perdedor",
                    "w_rank", "w_seed_principal", "w_seed_qualificatoria",
                    "duration", "duracao_min", "bracket", "fase").limit(10))

print("id_partida único?", base.select("id_partida").distinct().count() == base.count())

## 3. silver.partida — grão: uma partida

In [0]:
partida = base.select(
    "id_partida",
    F.col("circuit").alias("circuito"),
    F.col("tournament").alias("torneio"),
    F.col("country").alias("pais_torneio"),
    "ano", "data",
    F.col("gender").alias("genero"),
    "num_partida",
    F.col("bracket").alias("chave"),
    F.col("round").alias("rodada"),
    "fase",
    F.col("score").alias("placar"),
    "placar_regular", "sets_vencedor", "sets_perdedor",
    "duracao_min",
    "w_seed_principal", "w_seed_qualificatoria",
    "l_seed_principal", "l_seed_qualificatoria",
    "flag_partida_incompleta", "flag_duracao_suspeita", "flag_placar_inconsistente",
    F.current_timestamp().alias("_processado_em"),
)

(partida.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.partida"))

print(f"silver.partida: {spark.table('workspace.silver.partida').count():,} linhas")

## 4. silver.desempenho_atleta — grão: um atleta em uma partida

In [0]:
estatisticas = ["tot_attacks", "tot_kills", "tot_errors", "tot_hitpct",
                "tot_aces", "tot_serve_errors", "tot_blocks", "tot_digs"]

def posicao(lado, n):
    p = f"{lado}_p{n}"
    sel = base.select(
        "id_partida", "data", F.col("gender").alias("genero"),
        F.lit(p).alias("posicao"),
        F.lit(lado == "w").alias("vencedor"),
        F.regexp_replace(F.trim(F.col(f"{lado}_player{n}")), " +", " ").alias("nome"),
        F.to_date(F.col(f"{p}_birthdate")).alias("nascimento"),
        F.expr(f"try_cast({p}_hgt as int)").alias("altura_in"),
        F.col(f"{p}_country").alias("pais"),
        *[F.expr(f"try_cast({p}_{e} as double)").alias(e) for e in estatisticas],
    )
    return sel

desempenho = (
    posicao("w", 1).unionByName(posicao("w", 2))
    .unionByName(posicao("l", 1)).unionByName(posicao("l", 2))
    .withColumn("id_atleta", F.xxhash64(F.lower("nome"), "nascimento"))
    .withColumn("altura_cm", F.round(F.col("altura_in") * 2.54).cast("int"))
    .withColumn("idade_na_partida", F.floor(F.datediff("data", "nascimento") / 365.25).cast("int"))
    .withColumn("tem_estatistica", F.col("tot_attacks").isNotNull())
    .withColumn("flag_estatistica_invalida",
                (F.col("tot_attacks") < 0) | (F.col("tot_hitpct") > 1) | (F.col("tot_hitpct") < -1))
    .withColumn("flag_idade_atipica", (F.col("idade_na_partida") < 15) | (F.col("idade_na_partida") > 50))
    .drop("altura_in")
    .withColumn("_processado_em", F.current_timestamp())
)

(desempenho.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.desempenho_atleta"))

d = spark.table("workspace.silver.desempenho_atleta")
print(f"linhas: {d.count():,} (esperado 4 × 76.756 = 307.024)")
print(f"com estatística: {d.filter('tem_estatistica').count():,}")
print(f"estatística inválida: {d.filter('flag_estatistica_invalida').count()}")
print(f"idade atípica: {d.filter('flag_idade_atipica').count()}")

## 5. silver.atleta — grão: um atleta

In [0]:
atleta = (
    d.groupBy("id_atleta")
     .agg(
         F.mode("nome").alias("nome"),
         F.first("nascimento").alias("nascimento"),
         F.mode("genero").alias("genero"),
         F.mode("altura_cm").alias("altura_cm"),
         F.mode("pais").alias("pais"),
         F.min("data").alias("primeira_partida"),
         F.max("data").alias("ultima_partida"),
         F.count("*").alias("partidas"),
         F.sum(F.col("vencedor").cast("int")).alias("vitorias"),
     )
     .withColumn("flag_sem_nascimento", F.col("nascimento").isNull())
     .withColumn("_processado_em", F.current_timestamp())
)

(atleta.write.format("delta")
    .mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.atleta"))

a = spark.table("workspace.silver.atleta")
print(f"atletas: {a.count():,}")
print(f"sem nascimento: {a.filter('flag_sem_nascimento').count()}")
print(f"nomes que aparecem com mais de um nascimento: "
      f"{a.groupBy('nome').count().filter('count > 1').count()}")

In [0]:
# os 12 nomes com mais de um nascimento
display(
    a.join(a.groupBy("nome").count().filter("count > 1").select("nome"), "nome")
     .select("nome", "nascimento", "pais", "partidas", "primeira_partida", "ultima_partida")
     .orderBy("nome", "nascimento")
)

# nomes que aparecem com e sem nascimento (mesmo atleta dividido em dois ids?)
com = a.filter("nascimento is not null").select("nome").distinct()
sem = a.filter("nascimento is null").select("nome").distinct()
print("nomes presentes nas duas situações:", com.join(sem, "nome").count())

## 6. Validação pós-carga

In [0]:
p = spark.table("workspace.silver.partida")
d = spark.table("workspace.silver.desempenho_atleta")
a = spark.table("workspace.silver.atleta")

# chaves primárias únicas
assert p.select("id_partida").distinct().count() == p.count(), "id_partida duplicado"
assert a.select("id_atleta").distinct().count() == a.count(), "id_atleta duplicado"
assert d.select("id_partida", "posicao").distinct().count() == d.count(), "partida × posição duplicada"

# integridade: todo desempenho aponta para partida e atleta existentes
assert d.join(p, "id_partida", "left_anti").count() == 0, "desempenho sem partida"
assert d.join(a, "id_atleta", "left_anti").count() == 0, "desempenho sem atleta"

# nada foi excluído: 4 linhas de desempenho por partida
assert d.count() == 4 * p.count(), "unpivot perdeu linhas"

print(f"partida {p.count():,} | desempenho_atleta {d.count():,} | atleta {a.count():,} — tudo consistente")

## Resumo dos tratamentos

| Tratamento | Onde |
|---|---|
| `"NA"` → NULL em todas as colunas | as três tabelas |
| Tipagem: datas, inteiros, duração em minutos | partida, desempenho |
| Ranking `Q16` / `"10, Q1"` → `seed_principal` + `seed_qualificatoria` | partida |
| Placar parseado em sets; `flag_partida_incompleta` para forfeit/retired | partida |
| `fase` derivada de `bracket` (grupos / qualificatória / eliminatória) | partida |
| Unpivot das 4 posições; altura polegada → cm; idade recalculada de nascimento × data | desempenho |
| Flags `estatistica_invalida` (32) e `idade_atipica` (134), sem excluir | desempenho |
| Dedup de atleta por nome normalizado + nascimento; 791 sem nascimento | atleta |

**Limitação conhecida:** atleta com nascimento ausente em parte das partidas vira dois `id_atleta` (4 casos identificados).